# DataInsight | Tabular: ✏️ <project title>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trthminh/CO5177/blob/main/notebooks/tabular.ipynb)

| | |
|---|---|
| **Course** | Programming Foundations for Data Analysis and Visualization (CO5177), Semester 261, 2026–2027 |
| **Lecturer** | Lê Thành Sách, Faculty of Computer Science and Engineering, HCMUT – VNU-HCM |
| **Group** | DataInsight: Trương Thanh Minh (2580893), Tô Anh Phát (2580894), Trần Lê Hoàng Long (2570443) |
| **Owner of this notebook** | Trần Lê Hoàng Long (2570443) |
| **Dataset** | ✏️ [<dataset name>](<link>), <source>, <license> |
| **Task** | ✏️ Binary / multi-class classification of `<target>` from `<n>` features |

**Problem statement.** ✏️ One paragraph: what is predicted, from which inputs, why it matters, and which metric decides the best model.

**Table of contents**
1. [Setup](#s1)
2. [Data loading](#s2)
3. [Dataset description](#s3)
4. [Data quality checks](#s4)
5. [Exploratory data analysis (EDA)](#s5)
6. [Data preparation](#s6)
7. [Modeling](#s7)
8. [Evaluation on the test set](#s8)
9. [Inference on new data](#s9)
10. [Conclusion](#s10)

> **How to run.** Open in Google Colab → **Runtime → Run all**. The notebook installs missing libraries and downloads the data by itself; no manual upload, Drive mount or API key is needed.

> ### 🧭 How to use this template (delete this cell before submitting)
> 1. Copy this file to `notebooks/` with its final name (the Colab badge above already points there) and keep the section numbers and headings unchanged, so the three notebooks of the group read the same way.
> 2. Fill in **section 1.3 (Configuration)** first. The notebook refuses to run while a `"TODO"` value is left there.
> 3. Run all once. Every generic cell (missing values, duplicates, split, pipelines, metrics) adapts to your columns by itself.
> 4. Then work through every `✏️ TODO` (markdown) and `# TODO` (code): dataset-specific rules, EDA questions, observations with **numbers from your own output**.
> 5. Cell pattern used everywhere: **markdown "what and why" → code with comments → markdown "Observation / Conclusion"**. Never leave a figure without a written conclusion.
> 6. Before submitting, go through the checklist in section 10 and `notebooks/templates/README.md`.
>
> Template source: `notebooks/templates/tabular_template.ipynb`.

<a id="s1"></a>
## 1. Setup

### 1.1 Install missing libraries

Google Colab already ships most of these libraries. The cell below only installs a package when Python cannot find it, so Run all stays fast on Colab and still works on a fresh local environment. 

In [ ]:
import importlib.util
import subprocess
import sys

# import name -> pip requirement. Add a line here for every extra library you use.
REQUIRED_PACKAGES = {
    "numpy": "numpy",
    "pandas": "pandas>=2.0",
    "sklearn": "scikit-learn>=1.2",
    "plotly": "plotly>=5.15",
    "gdown": "gdown>=4.6",
    "kagglehub": "kagglehub",
}

# find_spec returns None when a module is not installed; nothing is imported yet, so no restart is needed.
missing = [spec for module, spec in REQUIRED_PACKAGES.items() if importlib.util.find_spec(module) is None]
if missing:
    print("Installing:", ", ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("All required packages are already installed.")

### 1.2 Imports

Grouped by role. Each scikit-learn import is used in exactly one step, named in the comment.

In [ ]:
import itertools
import urllib.request
import warnings
import zipfile
from pathlib import Path

import numpy as np                         # arrays, vectorised maths
import pandas as pd                        # tables: read, clean, group, split X / y
import plotly.express as px                # quick Plotly charts from DataFrames
import plotly.graph_objects as go          # fine-grained Plotly charts
import plotly.io as pio                    # global Plotly template
from plotly.subplots import make_subplots  # several panels in one figure

import sklearn
from sklearn.base import clone                                  # fresh, unfitted copy of a pipeline
from sklearn.compose import ColumnTransformer                   # different preprocessing per column group
from sklearn.dummy import DummyClassifier                       # baseline that ignores the features
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer                        # fill missing values with train statistics
from sklearn.inspection import permutation_importance           # model-agnostic feature importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, average_precision_score, balanced_accuracy_score,
                             classification_report, confusion_matrix, f1_score,
                             precision_recall_curve, roc_auc_score, roc_curve)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline                           # chain preprocessing + model
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings("ignore", category=FutureWarning)  # hide library deprecation notices only; keep real warnings
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:,.3f}".format)

print(f"numpy {np.__version__} | pandas {pd.__version__} | scikit-learn {sklearn.__version__}")

### 1.3 Configuration

**Everything that depends on the chosen dataset is set here and only here.** The rest of the notebook reads these names, so changing dataset means editing this one cell.

In [ ]:
# ---- Reproducibility --------------------------------------------------------------------------
SEED = 42                     # one seed for every random step (split, models, CV) -> Run all gives the same numbers
np.random.seed(SEED)

# ---- Data source (section 2) ------------------------------------------------------------------
DATA_DIR = Path("data")       # local cache, ignored by git
DATA_SOURCE = {
    "kind": "TODO",           # "url" | "gdrive" | "kaggle"
    "location": "TODO",       # direct URL | Google Drive file id | Kaggle handle "owner/dataset"
    "filename": "TODO.csv",   # data file to read after download (searched recursively)
    # "archive": "TODO.zip",  # uncomment when the url/gdrive link is a .zip archive
}
READ_KWARGS = {}              # extra pandas reader arguments, e.g. {"sep": ";", "encoding": "latin-1"}

# ---- Problem definition -----------------------------------------------------------------------
TARGET = "TODO"               # label column
ID_COLS = []                  # identifiers (customer_id, ...): kept for inspection, never used as features
DROP_COLS = []                # excluded on purpose (leakage, redundant, ...); justify each one in section 6.1
FORCE_CATEGORICAL = []        # numeric-looking codes that are really categories (zip code, product code, ...)
NA_TOKENS = ["?", "NA", "N/A", "n/a", "null", "NULL", ""]  # strings that mean "missing" in this file

# ---- Domain rules (sections 4.4 and 6.1) ------------------------------------------------------
VALID_RANGES = {}             # TODO: {"age": (18, 100), "income": (0, None)}; None = no bound. Domain knowledge, not statistics
SPELLING_MAPS = {}            # TODO: {"gender": {"m": "Male", "male": "Male", "f": "Female"}}; keys are stripped + lower-cased
TO_NUMERIC = []               # TODO: numeric columns stored as text ("1,234"), found in section 4.3

# ---- Split and preprocessing ------------------------------------------------------------------
TEST_SIZE, VAL_SIZE = 0.15, 0.15   # 70% train / 15% validation / 15% test
MIN_CATEGORY_FREQ = 0.01           # categories rarer than 1% of train are merged into one "infrequent" column

# Stop Run all early with a clear message instead of failing 20 cells later.
for _name, _value in {"DATA_SOURCE": DATA_SOURCE, "TARGET": TARGET}.items():
    assert "TODO" not in str(_value), f"Fill in {_name} in section 1.3 before running the notebook."

### 1.4 Plot style and shared helpers

All figures use **Plotly** (interactive: hover shows exact values, zoom works on long series). One shared template keeps colours and fonts identical across the three notebooks of the group; the palette is the same as in the Time series notebook.

Rules for every figure:
- Build it with `plotly.express` / `plotly.graph_objects`, then call `show_fig(fig, title)`. The helper numbers figures automatically (`Figure 1.`, `Figure 2.`, …), so numbering never breaks when a cell is added.
- The title states the **conclusion** ("Class 1 is only 12% of samples"), not the chart type ("Bar chart of classes").
- Label both axes with units, and follow every figure with a markdown observation that quotes numbers.

In [ ]:
# Colours by role, fixed order: categorical series take BLUE, ORANGE, AQUA, YELLOW, then RED/MUTED.
BLUE, ORANGE, AQUA, YELLOW = "#2a78d6", "#eb6834", "#1baf7a", "#eda100"
RED, MUTED, GRID = "#e34948", "#898781", "#e1e0d9"
QUALITATIVE = [BLUE, ORANGE, AQUA, YELLOW, RED, MUTED]
SEQ_BLUE = ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"]  # counts, intensities
DIVERGING = [[0, BLUE], [0.5, "#f0efec"], [1, RED]]                                          # correlations (-1..1)

# A named Plotly template = one place to change the look of every chart.
pio.templates["datainsight"] = go.layout.Template(layout=dict(
    font=dict(family="Inter, Segoe UI, Arial, sans-serif", size=13, color="#0b0b0b"),
    colorway=QUALITATIVE,
    paper_bgcolor="#ffffff", plot_bgcolor="#ffffff",
    title=dict(x=0.01, font=dict(size=16)),
    xaxis=dict(gridcolor=GRID, linecolor="#c9c8c3", zeroline=False, title_font_color="#52514e"),
    yaxis=dict(gridcolor=GRID, linecolor="#c9c8c3", zeroline=False, title_font_color="#52514e"),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, x=0, title_text=""),
    margin=dict(l=60, r=20, t=70, b=50),
))
pio.templates.default = "plotly_white+datainsight"

_figure_number = itertools.count(1)  # global counter: Run all numbers figures 1, 2, 3, ... in order


def show_fig(fig, title, height=420, **layout):
    """Give the figure its number and conclusion-style title, then display it."""
    fig.update_layout(title=f"Figure {next(_figure_number)}. {title}", height=height, **layout)
    fig.show()


def hist_bar(values, bins=40, color=BLUE, name=None):
    """Histogram pre-computed with NumPy: the notebook stores ~40 bars instead of every data point."""
    values = np.asarray(values, dtype=float)
    counts, edges = np.histogram(values[~np.isnan(values)], bins=bins)
    centers = (edges[:-1] + edges[1:]) / 2
    return go.Bar(x=centers, y=counts, width=np.diff(edges) * 0.92, marker_color=color, name=name,
                  hovertemplate="%{x:,.2f}: %{y:,}<extra></extra>")


def grid_position(i, ncols):
    """(row, col) in a make_subplots grid for the i-th panel, filled left to right."""
    return i // ncols + 1, i % ncols + 1

<a id="s2"></a>
## 2. Data loading

### 2.1 Download

The data is **never uploaded by hand**: the notebook fetches it from a public source and caches it in `data/` (ignored by git), so the lecturer only has to press Run all. `download_dataset` supports three kinds of source:

| `kind` | `location` | Library | Notes |
|---|---|---|---|
| `"url"` | direct download link | `urllib` (standard library) | GitHub raw, UCI, Hugging Face `resolve/...` links, … |
| `"gdrive"` | Google Drive **file id** (the part after `/d/` in the share link) | `gdown` | Share the file as *Anyone with the link → Viewer* |
| `"kaggle"` | dataset handle `owner/dataset-name` | `kagglehub` | Public datasets need no API key |

If the link points to a `.zip` archive, also set `"archive"`; the archive is extracted and `filename` is searched inside it.

In [ ]:
def download_dataset(source, data_dir=DATA_DIR):
    """Download one data source (skipped if already cached) and return the path of `source["filename"]`."""
    data_dir.mkdir(parents=True, exist_ok=True)
    kind, location, filename = source["kind"], source["location"], source["filename"]

    if kind == "kaggle":
        import kagglehub
        root = Path(kagglehub.dataset_download(location))  # kagglehub keeps its own cache
    else:
        root = data_dir
        archive = source.get("archive")
        target = data_dir / (archive or filename)
        if not target.exists():  # cache: never download twice
            if kind == "url":
                urllib.request.urlretrieve(location, target)
            elif kind == "gdrive":
                import gdown
                gdown.download(id=location, output=str(target), quiet=False)
            else:
                raise ValueError(f"Unknown source kind {kind!r}: use 'url', 'gdrive' or 'kaggle'.")
        if archive:
            with zipfile.ZipFile(target) as zf:
                zf.extractall(data_dir)

    matches = sorted(root.rglob(filename))  # rglob: the file may sit in a sub-folder of the archive
    if not matches:
        found = [p.relative_to(root).as_posix() for p in root.rglob("*") if p.is_file()][:20]
        raise FileNotFoundError(f"{filename!r} not found under {root}. Files found: {found}")
    print(f"Data file: {matches[0]} ({matches[0].stat().st_size / 1e6:,.1f} MB)")
    return matches[0]


def read_table(path, na_values=None, **kwargs):
    """Read a data file with the pandas reader that matches its extension."""
    suffix = path.suffix.lower()
    if suffix in {".csv", ".txt", ".data"}:
        return pd.read_csv(path, na_values=na_values, **kwargs)
    if suffix == ".tsv":
        return pd.read_csv(path, sep="\t", na_values=na_values, **kwargs)
    if suffix in {".xlsx", ".xls"}:
        return pd.read_excel(path, na_values=na_values, **kwargs)
    if suffix == ".parquet":
        return pd.read_parquet(path, **kwargs)
    if suffix in {".json", ".jsonl"}:
        return pd.read_json(path, lines=suffix == ".jsonl", **kwargs)
    raise ValueError(f"No reader configured for {suffix!r}; add one in read_table.")

### 2.2 Read the table

In [ ]:
data_path = download_dataset(DATA_SOURCE)
raw = read_table(data_path, na_values=NA_TOKENS, **READ_KWARGS)
raw.columns = raw.columns.str.strip()  # header names with stray spaces break column look-ups later
assert TARGET in raw.columns, f"TARGET {TARGET!r} not found. Columns: {list(raw.columns)}"

print(f"Shape: {raw.shape[0]:,} rows x {raw.shape[1]} columns")
raw.head()

### 2.3 Does the dataset meet the assignment requirements?

Assignment §4.2 asks for a tabular dataset with ≥ 2,000 samples, ≥ 10 columns, missing values, categorical and numerical columns. The table is the evidence for the report.

In [ ]:
_features = [c for c in raw.columns if c not in [TARGET, *ID_COLS]]
_n_numeric = len(raw[_features].select_dtypes("number").columns)
requirements = pd.DataFrame([
    ("At least 2,000 samples", f"{len(raw):,} rows", len(raw) >= 2000),
    ("At least 10 columns", f"{raw.shape[1]} columns", raw.shape[1] >= 10),
    ("Has missing values", f"{int(raw.isna().sum().sum()):,} missing cells", bool(raw.isna().any().any())),
    ("Has categorical columns", f"{len(_features) - _n_numeric} non-numeric columns", len(_features) > _n_numeric),
    ("Has numerical columns", f"{_n_numeric} numeric columns", _n_numeric > 0),
], columns=["Requirement (assignment §4.2)", "This dataset", "Met"])
requirements

> ✏️ **TODO:** If a requirement is not met, explain here why the dataset is still suitable (e.g. missing values appear only as `"?"`, which section 4.1 detects).

<a id="s3"></a>
## 3. Dataset description

✏️ **TODO:** Origin of the data (who collected it, when, how), what one row represents, licence.

| Column | Type | Meaning | Role |
|---|---|---|---|
| ✏️ `col_a` | numeric | … | feature |
| ✏️ `col_b` | categorical | … | feature |
| ✏️ `target` | categorical | … | **target** |

In [ ]:
raw.info()  # dtype per column: object = text, int64/float64 = numbers; a numeric column with NaN becomes float64

### 3.1 Column roles

Columns are split automatically by dtype; `FORCE_CATEGORICAL` overrides codes that look numeric. These two lists drive the plots in sections 4–5 and the preprocessing in section 6.

In [ ]:
def column_roles(df):
    """Split feature columns into numeric and categorical lists, honouring the configuration."""
    features = [c for c in df.columns if c not in {TARGET, *ID_COLS, *DROP_COLS}]
    numeric = [c for c in features if pd.api.types.is_numeric_dtype(df[c]) and c not in FORCE_CATEGORICAL]
    categorical = [c for c in features if c not in numeric]
    return features, numeric, categorical


FEATURES, NUM_COLS, CAT_COLS = column_roles(raw)
print(f"{len(NUM_COLS)} numeric:", NUM_COLS)
print(f"{len(CAT_COLS)} categorical:", CAT_COLS)

In [ ]:
raw[NUM_COLS].describe().T  # count < number of rows => missing values; min/max reveal impossible values

In [ ]:
cat_summary = pd.DataFrame({
    "distinct values": raw[CAT_COLS].nunique(),
    "most frequent": raw[CAT_COLS].mode().iloc[0] if CAT_COLS else [],
    "share of most frequent": [raw[c].value_counts(normalize=True).iloc[0] for c in CAT_COLS],
    "missing": raw[CAT_COLS].isna().sum(),
})
cat_summary  # many distinct values = high cardinality -> one-hot would create many columns (see MIN_CATEGORY_FREQ)

**Observation.** ✏️ Number of rows/columns, which columns are numeric vs categorical, any column whose dtype is surprising, high-cardinality columns.

<a id="s4"></a>
## 4. Data quality checks

This section **only diagnoses**; every fix is applied in section 6.1 so the cleaning rules sit in one function. Following the course, four kinds of error need four different fixes: missing values, impossible values, duplicates and inconsistent spelling. Class imbalance and outliers are checked too because they change the modeling choices.

### 4.1 Missing values

Two counts: real `NaN`, and **hidden** missing values: tokens such as `" ?"` that `na_values` did not catch because of surrounding spaces.

In [ ]:
text_cols = raw.columns.difference(raw.select_dtypes("number").columns)
hidden = {c: int(raw[c].dropna().astype(str).str.strip().isin(NA_TOKENS).sum()) for c in text_cols}

missing = pd.DataFrame({"NaN": raw.isna().sum(), "hidden tokens": pd.Series(hidden)}).fillna(0).astype(int)
missing["total"] = missing["NaN"] + missing["hidden tokens"]
missing["% of rows"] = missing["total"] / len(raw) * 100
missing = missing[missing["total"] > 0].sort_values("total", ascending=False)

if missing.empty:
    print("No missing values found.")
else:
    fig = go.Figure(go.Bar(x=missing.index, y=missing["% of rows"], marker_color=BLUE,
                           text=[f"{p:.1f}%" for p in missing["% of rows"]], textposition="outside",
                           hovertemplate="%{x}: %{y:.2f}% of rows<extra></extra>"))
    show_fig(fig, f"{len(missing)} columns contain missing values (max {missing['% of rows'].max():.1f}% of rows)",
             xaxis_title="Column", yaxis_title="% of rows missing")
missing

**Is missingness related to the target?** If the class distribution differs between rows where a value is missing and rows where it is present, the fact of being missing carries information (data is not missing completely at random).

In [ ]:
target_share = raw[TARGET].value_counts(normalize=True)
MINORITY = target_share.idxmin()  # the class we care most about when classes are imbalanced

rows = []
for c in missing.index:
    is_na = raw[c].isna() | raw[c].astype(str).str.strip().isin(NA_TOKENS)
    if c == TARGET or is_na.all():
        continue
    rows.append({"column": c,
                 f"% '{MINORITY}' when missing": (raw.loc[is_na, TARGET] == MINORITY).mean() * 100,
                 f"% '{MINORITY}' when present": (raw.loc[~is_na, TARGET] == MINORITY).mean() * 100})
pd.DataFrame(rows)

**Observation.** ✏️ Which columns are missing, how much, whether hidden tokens exist, and whether missingness relates to the target. Decide: impute (median / most frequent, fitted on train in section 6.3), drop the column (> ~40–50% missing), or drop rows (only if very few).

### 4.2 Duplicates

- **Exact duplicates** inflate some patterns and can put the same sample in both train and test (leakage).
- **Same features, different labels** are conflicting samples: no model can get all of them right, so they cap the achievable accuracy.

In [ ]:
non_id = [c for c in raw.columns if c not in ID_COLS]
feature_like = [c for c in non_id if c != TARGET]

dup_table = pd.DataFrame({
    "check": ["Exact duplicate rows", "Duplicates ignoring ID columns", "Repeated ID values",
              "Same features, different label (groups)"],
    "count": [
        int(raw.duplicated().sum()),
        int(raw.duplicated(subset=non_id).sum()),
        int(raw.duplicated(subset=ID_COLS).sum()) if ID_COLS else 0,
        int((raw.groupby(feature_like, dropna=False)[TARGET].nunique() > 1).sum()),
    ],
})
dup_table

**Observation.** ✏️ How many duplicates; whether they are real repeated records or a data-collection artefact; decision (drop exact duplicates before splitting).

### 4.3 Inconsistent spelling and numbers stored as text

`"HCM"`, `" HCM"` and `"hcm"` are three different categories for pandas. The table compares the number of distinct values before and after `strip().lower()`; a drop means variants of the same value exist. It also flags text columns whose values are mostly numbers.

In [ ]:
rows, variant_examples = [], {}
for c in CAT_COLS:
    s = raw[c].dropna().astype(str)
    norm = s.str.strip().str.lower()
    rows.append({"column": c,
                 "distinct (raw)": s.nunique(),
                 "distinct (strip + lower)": norm.nunique(),
                 "values with spaces at the ends": int((s != s.str.strip()).sum()),
                 "share parseable as number": pd.to_numeric(s.str.replace(",", ""), errors="coerce").notna().mean()})
    groups = s.groupby(norm).unique()
    variants = groups[groups.map(len) > 1]
    if len(variants):
        variant_examples[c] = variants.head(5).map(list).to_dict()

display(pd.DataFrame(rows))
for c, examples in variant_examples.items():
    print(f"{c}: {examples}")

**Observation.** ✏️ Columns with spelling variants (→ add them to `SPELLING_MAPS`) and text columns that should be numeric (→ add them to `TO_NUMERIC`).

### 4.4 Impossible values

Values that cannot be true (age 200, negative income) are **errors**, unlike outliers which can be real. The rules come from domain knowledge and are written in `VALID_RANGES`.

In [ ]:
rows = []
for c, (low, high) in VALID_RANGES.items():
    values = pd.to_numeric(raw[c], errors="coerce")
    too_low = int((values < low).sum()) if low is not None else 0
    too_high = int((values > high).sum()) if high is not None else 0
    rows.append({"column": c, "rule": f"[{low}, {high}]", "below": too_low, "above": too_high,
                 "observed min": values.min(), "observed max": values.max()})
pd.DataFrame(rows) if rows else print("VALID_RANGES is empty: add domain rules in section 1.3.")

**Observation.** ✏️ Which rules are violated and how often. Decision: drop the rows (as in the course) or set the value to NaN and impute.

### 4.5 Outliers (IQR rule)

A value is flagged when it lies outside $[Q_1 - 1.5\,IQR,\; Q_3 + 1.5\,IQR]$ with $IQR = Q_3 - Q_1$. Flagged values are **not errors by default**: a very high income can be real. The box plots show how far they go.

In [ ]:
q1, q3 = raw[NUM_COLS].quantile(0.25), raw[NUM_COLS].quantile(0.75)
iqr = q3 - q1
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
outliers = pd.DataFrame({"lower fence": lower, "upper fence": upper,
                         "n below": (raw[NUM_COLS] < lower).sum(), "n above": (raw[NUM_COLS] > upper).sum()})
outliers["% of rows"] = (outliers["n below"] + outliers["n above"]) / len(raw) * 100
outliers.sort_values("% of rows", ascending=False)

In [ ]:
ncols = 3
plot_cols = NUM_COLS[:12]  # keep the figure readable; choose the most relevant columns if there are more
nrows = max(1, -(-len(plot_cols) // ncols))  # ceiling division
fig = make_subplots(rows=nrows, cols=ncols, subplot_titles=plot_cols, vertical_spacing=0.12)
for i, c in enumerate(plot_cols):
    r, k = grid_position(i, ncols)
    fig.add_trace(go.Box(y=raw[c], name=c, marker_color=BLUE, boxpoints="outliers", showlegend=False), row=r, col=k)
show_fig(fig, "Spread and outliers of the numeric columns (each panel has its own scale)", height=300 * nrows)

**Observation.** ✏️ Which columns have heavy tails; real values or errors? Decision: keep (tree models are robust), cap/clip, log-transform, or treat as errors in 4.4.

### 4.6 Class balance

Accuracy is misleading when one class dominates: predicting the majority class always already scores its share. This decides the metrics (F1-macro, balanced accuracy) and whether to use `class_weight="balanced"`.

In [ ]:
counts = raw[TARGET].value_counts(dropna=False)
shares = counts / counts.sum() * 100
fig = go.Figure(go.Bar(x=counts.index.astype(str), y=counts.values, marker_color=BLUE,
                       text=[f"{n:,} ({p:.1f}%)" for n, p in zip(counts, shares)], textposition="outside",
                       hovertemplate="%{x}: %{y:,} rows<extra></extra>"))
imbalance_ratio = counts.max() / counts.min()
show_fig(fig, f"Class balance of '{TARGET}': majority/minority ratio = {imbalance_ratio:.1f}",
         xaxis_title=TARGET, yaxis_title="Rows")
print(f"Accuracy of always predicting the majority class: {shares.max():.1f}%")

**Observation.** ✏️ Class shares, imbalance ratio, consequence for metrics and models.

<a id="s5"></a>
## 5. Exploratory data analysis (EDA)

Every subsection answers a question. The required views come first (distributions, relation to the target, correlation); section 5.5 holds the **question-driven analyses** that earn creativity points (rubric §7.1: a figure that answers a question, with a caption stating the conclusion and numbers to compare).

### 5.1 How are the numeric features distributed?

In [ ]:
ncols = 3
plot_cols = NUM_COLS[:12]
nrows = max(1, -(-len(plot_cols) // ncols))
fig = make_subplots(rows=nrows, cols=ncols, subplot_titles=plot_cols, vertical_spacing=0.12)
for i, c in enumerate(plot_cols):
    r, k = grid_position(i, ncols)
    fig.add_trace(hist_bar(raw[c], bins=40), row=r, col=k)
fig.update_layout(showlegend=False)
show_fig(fig, "Distribution of the numeric features", height=280 * nrows)
raw[NUM_COLS].skew().sort_values(key=abs, ascending=False).rename("skewness").to_frame()  # |skew| > 1: strongly asymmetric

**Observation.** ✏️ Shape (symmetric, skewed, bimodal), spikes at special values (0, 99, …), skewness numbers.

### 5.2 Which categories dominate each categorical feature?

In [ ]:
TOP_K = 10
ncols = 2
plot_cols = CAT_COLS[:8]
nrows = max(1, -(-len(plot_cols) // ncols))
fig = make_subplots(rows=nrows, cols=ncols, subplot_titles=plot_cols, vertical_spacing=0.14)
for i, c in enumerate(plot_cols):
    vc = raw[c].value_counts(dropna=False)
    top = vc.head(TOP_K)
    if len(vc) > TOP_K:  # merge the long tail so the panel stays readable
        top = pd.concat([top, pd.Series({"(other)": vc.iloc[TOP_K:].sum()})])
    r, k = grid_position(i, ncols)
    fig.add_trace(go.Bar(x=top.index.astype(str), y=top.values, marker_color=BLUE, showlegend=False,
                         hovertemplate="%{x}: %{y:,}<extra></extra>"), row=r, col=k)
show_fig(fig, f"Top {TOP_K} categories per categorical feature", height=320 * nrows)

**Observation.** ✏️ Dominant and rare categories, high cardinality, values that need merging.

### 5.3 Which numeric features separate the classes?

Box plots per class: the less the boxes overlap, the more useful the feature. The table quantifies it with the median per class.

In [ ]:
ncols = 3
plot_cols = NUM_COLS[:9]
classes = raw[TARGET].dropna().unique()
nrows = max(1, -(-len(plot_cols) // ncols))
fig = make_subplots(rows=nrows, cols=ncols, subplot_titles=plot_cols, vertical_spacing=0.12)
for i, c in enumerate(plot_cols):
    r, k = grid_position(i, ncols)
    for j, cls in enumerate(classes):
        fig.add_trace(go.Box(y=raw.loc[raw[TARGET] == cls, c], name=str(cls), marker_color=QUALITATIVE[j % 6],
                             boxpoints=False, showlegend=(i == 0), legendgroup=str(cls)), row=r, col=k)
show_fig(fig, f"Numeric features by '{TARGET}' class", height=300 * nrows)
raw.groupby(TARGET)[NUM_COLS].median().T  # median per class: large gaps = discriminative feature

**Observation.** ✏️ The 2–3 features whose medians differ most between classes, with numbers.

### 5.4 How does the minority-class rate change across categories?

For each categorical feature: share of the minority class per category (top categories by count). The dotted line is the overall rate; bars far from it mark informative categories.

In [ ]:
overall_rate = (raw[TARGET] == MINORITY).mean() * 100
ncols = 2
plot_cols = CAT_COLS[:8]
nrows = max(1, -(-len(plot_cols) // ncols))
fig = make_subplots(rows=nrows, cols=ncols, subplot_titles=plot_cols, vertical_spacing=0.14)
for i, c in enumerate(plot_cols):
    top_categories = raw[c].value_counts().head(10).index
    rate = (raw[raw[c].isin(top_categories)].groupby(c)[TARGET]
            .apply(lambda s: (s == MINORITY).mean() * 100).sort_values())
    r, k = grid_position(i, ncols)
    fig.add_trace(go.Bar(x=rate.index.astype(str), y=rate.values, marker_color=ORANGE, showlegend=False,
                         hovertemplate="%{x}: %{y:.1f}%<extra></extra>"), row=r, col=k)
    fig.add_hline(y=overall_rate, line_dash="dot", line_color=MUTED, row=r, col=k)
show_fig(fig, f"Share of '{MINORITY}' per category (dotted line = overall {overall_rate:.1f}%)", height=320 * nrows)

**Observation.** ✏️ Categories with the highest / lowest minority rate and how far they are from the overall rate.

### 5.5 Are numeric features correlated with each other?

Pearson correlation measures **linear** association. Pairs with |r| > 0.9 carry almost the same information: linear models cannot separate their effects, so keep one of them.

In [ ]:
corr = raw[NUM_COLS].corr()
fig = go.Figure(go.Heatmap(z=corr.values, x=corr.columns, y=corr.columns, zmin=-1, zmax=1, colorscale=DIVERGING,
                           text=corr.round(2).values, texttemplate="%{text}", xgap=2, ygap=2,
                           colorbar=dict(title="r"), hovertemplate="%{y} x %{x}: %{z:.2f}<extra></extra>"))
show_fig(fig, "Pearson correlation between numeric features", height=max(420, 40 * len(NUM_COLS)),
         yaxis_autorange="reversed")

pairs = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1)).stack()  # upper triangle: each pair once
pairs[pairs.abs() > 0.9].rename("r").to_frame() if (pairs.abs() > 0.9).any() else print("No pair with |r| > 0.9.")

**Observation.** ✏️ Strongly correlated pairs and which one is dropped (add it to `DROP_COLS` and explain).

### 5.6 Question 1: ✏️ <write the question, e.g. "Does the minority rate depend on X and Y together?">

Pattern for every question: **question → figure with a conclusion title → numbers → conclusion**. The example below shows the minority-class rate over two features at once (a numeric feature is cut into quantile bins); replace it with the question that matters for your data.

In [ ]:
Q1_ROW, Q1_COL = "TODO", "TODO"  # TODO: two feature names


def as_groups(series, bins=5):
    """Numeric -> quantile bins (equal-sized groups); categorical -> unchanged."""
    return pd.qcut(series, q=bins, duplicates="drop").astype(str) if pd.api.types.is_numeric_dtype(series) else series


if "TODO" not in (Q1_ROW, Q1_COL):
    table = (raw.assign(_row=as_groups(raw[Q1_ROW]), _col=as_groups(raw[Q1_COL]), _hit=raw[TARGET] == MINORITY)
                .pivot_table(index="_row", columns="_col", values="_hit", aggfunc="mean") * 100)
    fig = go.Figure(go.Heatmap(z=table.values, x=table.columns.astype(str), y=table.index.astype(str),
                               colorscale=SEQ_BLUE, text=table.round(1).values, texttemplate="%{text}%",
                               colorbar=dict(title="%"), hovertemplate=f"{Q1_ROW}=%{{y}}, {Q1_COL}=%{{x}}: %{{z:.1f}}%<extra></extra>"))
    show_fig(fig, f"✏️ <conclusion>: share of '{MINORITY}' by {Q1_ROW} and {Q1_COL}",
             xaxis_title=Q1_COL, yaxis_title=Q1_ROW, height=460)

**Conclusion.** ✏️ Answer the question with the numbers from the figure.

### 5.7 Question 2: ✏️ <question>

✏️ Add as many questions as needed, each with the same pattern. Ideas from the rubric: target vs 2–3 key features, effect of outliers on the target, a sub-group that behaves differently.

In [ ]:
# TODO: analysis for question 2 (compute -> figure with show_fig -> print the numbers quoted in the conclusion).

**Conclusion.** ✏️

<a id="s6"></a>
## 6. Data preparation

The order follows the course (*Pandas* module, "fit on train, transform the rest"):

| When | What | Why |
|---|---|---|
| **Before** the split, on the whole table | strip text, map spellings, convert text→number, drop rows without label, drop exact duplicates, drop impossible values | Row-by-row rules: they use no statistics of other rows, so they cannot leak information. Removing duplicates first keeps the same sample out of both train and test. |
| **After** the split, fitted on **train only** | median / most frequent for missing values, mean & std for scaling, list of categories for one-hot | These are statistics of the data. Fitted on the whole table they would "see" validation and test rows (leakage). |

### 6.1 Cleaning rules (before the split)

In [ ]:
def normalize_values(df):
    """Row-wise text fixes shared by training data and new data (section 9)."""
    out = df.copy()
    for c in out.columns.difference(out.select_dtypes("number").columns):
        s = out[c]
        s = s.where(s.isna(), s.astype(str).str.strip())   # strip real values only; NaN stays NaN
        out[c] = s.replace(NA_TOKENS, np.nan)              # hidden missing tokens -> NaN
    for c, mapping in SPELLING_MAPS.items():
        lowered = out[c].str.lower()
        out[c] = lowered.map(mapping).fillna(out[c])       # mapped variants -> canonical; others unchanged
    for c in TO_NUMERIC:
        out[c] = pd.to_numeric(out[c].astype(str).str.replace(",", ""), errors="coerce")
    return out


def clean_table(df):
    """Apply every rule that is safe before splitting and report the row count after each step."""
    steps = [("raw", len(df))]
    out = normalize_values(df)
    out = out.drop(columns=DROP_COLS)
    out = out.dropna(subset=[TARGET])                      # a row without label cannot be used for training or testing
    steps.append(("label present", len(out)))
    out = out.drop_duplicates()
    steps.append(("exact duplicates removed", len(out)))
    valid = pd.Series(True, index=out.index)
    for c, (low, high) in VALID_RANGES.items():
        ok = out[c].isna()                                 # a missing value is not invalid: the imputer handles it
        ok |= out[c].between(-np.inf if low is None else low, np.inf if high is None else high)
        valid &= ok
    out = out.loc[valid]
    steps.append(("impossible values removed", len(out)))
    return out.reset_index(drop=True), pd.DataFrame(steps, columns=["step", "rows"])


df, clean_steps = clean_table(raw)
FEATURES, NUM_COLS, CAT_COLS = column_roles(df)  # dtypes may have changed (TO_NUMERIC), so recompute the roles
print("Numeric:", NUM_COLS)
print("Categorical:", CAT_COLS)
clean_steps.assign(removed=lambda t: t["rows"].shift(1) - t["rows"])

**Observation.** ✏️ Rows removed per step and why; justification of every column in `DROP_COLS`.

### 6.2 Train / validation / test split

- **Train (70%)** fits the preprocessing and the models; **validation (15%)** compares models and tunes them; **test (15%)** is used **once**, in section 8, to report the final score.
- `stratify` keeps the class proportions identical in the three parts; `random_state=SEED` makes the split reproducible.
- The label is encoded as integers 0, 1, … (`LabelEncoder`) so every metric works the same way for binary and multi-class targets.

> ✏️ If one entity (customer, patient) can appear in several rows, split by entity instead (`GroupShuffleSplit`) so the same entity is never in both train and test.

In [ ]:
X = df[FEATURES]
MINORITY = df[TARGET].value_counts().idxmin()      # recomputed: cleaning may have changed the label spelling
label_encoder = LabelEncoder().fit(df[TARGET])
y = label_encoder.transform(df[TARGET])            # class names -> 0..K-1, in alphabetical order
CLASS_NAMES = [str(c) for c in label_encoder.classes_]
POS = CLASS_NAMES.index(str(MINORITY))             # index of the minority class (the "positive" class in binary tasks)
IS_BINARY = len(CLASS_NAMES) == 2

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=TEST_SIZE + VAL_SIZE, stratify=y, random_state=SEED)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=TEST_SIZE / (TEST_SIZE + VAL_SIZE), stratify=y_temp, random_state=SEED)

split_table = pd.DataFrame({
    "set": ["train", "validation", "test"],
    "rows": [len(X_train), len(X_val), len(X_test)],
    "share": [len(p) / len(X) for p in (X_train, X_val, X_test)],
    f"% '{MINORITY}'": [np.mean(p == POS) * 100 for p in (y_train, y_val, y_test)],
})
split_table

### 6.3 Encoding and scaling, fitted on train only

| Column type | Step 1 | Step 2 |
|---|---|---|
| numeric | `SimpleImputer(strategy="median")`: the median is robust to outliers | `StandardScaler`: $z = (x - \mu_{train}) / \sigma_{train}$, needed by linear models; trees ignore it |
| categorical | `SimpleImputer(strategy="most_frequent")` | `OneHotEncoder`: one 0/1 column per category. Integer codes would make the model treat categories as ordered magnitudes. `handle_unknown="infrequent_if_exist"` maps categories never seen in train to the infrequent column instead of crashing; `min_frequency` merges rare categories. |

`ColumnTransformer` routes each column group to its pipeline. The **unfitted** `preprocessor` is placed inside every model pipeline in section 7, so `fit` learns its statistics from the training rows only, also inside cross-validation.

In [ ]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=MIN_CATEGORY_FREQ,
                              sparse_output=False)),
])
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, NUM_COLS),
    ("cat", categorical_pipeline, CAT_COLS),
])

# Fit once here only to inspect the result; the models below refit their own copy.
prep_check = clone(preprocessor).fit(X_train)                 # learns medians, modes, means, stds, category lists
X_train_ml = prep_check.transform(X_train)
X_val_ml, X_test_ml = prep_check.transform(X_val), prep_check.transform(X_test)
feature_names = prep_check.get_feature_names_out()

print(f"train {X_train_ml.shape} | validation {X_val_ml.shape} | test {X_test_ml.shape}")
print(f"{len(NUM_COLS)} numeric + {len(CAT_COLS)} categorical columns -> {len(feature_names)} model columns")
assert not np.isnan(X_train_ml).any() and not np.isnan(X_test_ml).any(), "NaN left after preprocessing"
pd.Series(prep_check.named_transformers_["num"].named_steps["imputer"].statistics_, index=NUM_COLS,
          name="median learned on train").to_frame()

### 6.4 Before / after scaling

Scaling changes the **scale** (mean 0, standard deviation 1), not the **shape** of a distribution. The figure checks it on the two numeric columns with the largest spread.

In [ ]:
show_cols = X_train[NUM_COLS].std().sort_values(ascending=False).index[:2].tolist()
fig = make_subplots(rows=len(show_cols), cols=2, vertical_spacing=0.2,
                    subplot_titles=[t for c in show_cols for t in (f"{c}: before", f"{c}: after StandardScaler")])
for r, c in enumerate(show_cols, start=1):
    j = list(feature_names).index(f"num__{c}")
    fig.add_trace(hist_bar(X_train[c], color=BLUE), row=r, col=1)
    fig.add_trace(hist_bar(X_train_ml[:, j], color=ORANGE), row=r, col=2)
fig.update_layout(showlegend=False)
show_fig(fig, "StandardScaler changes the scale, not the shape of the distribution", height=300 * len(show_cols))

<a id="s7"></a>
## 7. Modeling

### 7.1 Compare models on the validation set

Each model is a `Pipeline(preprocessor → model)`, so preprocessing is always fitted on the data the model is fitted on.

| Model | Why it is in the comparison |
|---|---|
| `DummyClassifier` | **Baseline**: always predicts the majority class. A useful model must beat it clearly. |
| `LogisticRegression` | Linear, fast, interpretable coefficients. |
| `DecisionTreeClassifier` | Non-linear rules, easy to explain; `max_depth` limits overfitting. |
| `RandomForestClassifier` | Average of many trees: more stable than a single tree. |
| `HistGradientBoostingClassifier` | Trees built one after another, each correcting the previous ones; strong on tabular data. |
| `class_weight="balanced"` variants | Give minority-class errors a larger weight; compared with the plain version to measure the effect of handling imbalance. |

The main metric is **F1-macro** (average F1 over classes, so each class counts equally, so it is robust to imbalance). Accuracy, balanced accuracy and ROC-AUC are reported alongside.

In [ ]:
def evaluate(model, X, y):
    """Classification metrics of a fitted model on one data split."""
    pred = model.predict(X)
    scores = {"accuracy": accuracy_score(y, pred),
              "balanced_accuracy": balanced_accuracy_score(y, pred),
              "f1_macro": f1_score(y, pred, average="macro")}
    proba = model.predict_proba(X)
    scores["roc_auc"] = (roc_auc_score(y == POS, proba[:, POS]) if IS_BINARY
                         else roc_auc_score(y, proba, multi_class="ovr"))
    return scores


def make_model(estimator):
    return Pipeline([("prep", clone(preprocessor)), ("model", estimator)])


MODELS = {
    "Baseline (majority class)": DummyClassifier(strategy="most_frequent"),
    "Logistic Regression": LogisticRegression(max_iter=2000),
    "Logistic Regression (balanced)": LogisticRegression(max_iter=2000, class_weight="balanced"),
    "Decision Tree": DecisionTreeClassifier(max_depth=8, random_state=SEED),
    "Random Forest": RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=SEED),
    "Random Forest (balanced)": RandomForestClassifier(n_estimators=300, class_weight="balanced",
                                                       n_jobs=-1, random_state=SEED),
    "HistGradientBoosting": HistGradientBoostingClassifier(random_state=SEED),
}

fitted, rows = {}, []
for name, estimator in MODELS.items():
    model = make_model(estimator).fit(X_train, y_train)
    fitted[name] = model
    rows.append({"model": name, **evaluate(model, X_val, y_val)})
    print(f"done: {name}")

results = pd.DataFrame(rows).set_index("model").sort_values("f1_macro", ascending=False)
results

In [ ]:
plot_df = results.reset_index().melt(id_vars="model", var_name="metric", value_name="score")
fig = px.bar(plot_df, x="score", y="model", color="metric", orientation="h", barmode="group",
             category_orders={"model": results.index.tolist()[::-1]})
fig.update_xaxes(range=[0, 1])
show_fig(fig, f"Validation scores: best F1-macro = {results['f1_macro'].max():.3f} ({results['f1_macro'].idxmax()})",
         height=500, xaxis_title="Score on validation", yaxis_title="")

**Observation.** ✏️ Ranking of models; gap to the baseline; effect of `class_weight="balanced"` (plain vs balanced variant).

### 7.2 Hyperparameter tuning

`GridSearchCV` tries every combination in `PARAM_GRID` with stratified k-fold **cross-validation on the training set** (validation and test stay untouched). Parameters are prefixed with `model__` because the estimator is the `"model"` step of the pipeline.

In [ ]:
TUNE_MODEL = "Random Forest (balanced)"  # TODO: usually the best family from 7.1
PARAM_GRID = {                           # TODO: adapt to TUNE_MODEL (parameter names differ between models)
    "model__max_depth": [None, 10, 20],
    "model__min_samples_leaf": [1, 5],
}

search = GridSearchCV(make_model(MODELS[TUNE_MODEL]), PARAM_GRID, scoring="f1_macro",
                      cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED), n_jobs=-1)
search.fit(X_train, y_train)

tuned_name = f"{TUNE_MODEL} (tuned)"
fitted[tuned_name] = search.best_estimator_
results.loc[tuned_name] = pd.Series(evaluate(search.best_estimator_, X_val, y_val))  # new row, aligned by column name
print("Best parameters:", search.best_params_, f"| CV F1-macro = {search.best_score_:.3f}")
(pd.DataFrame(search.cv_results_)[["params", "mean_test_score", "std_test_score", "rank_test_score"]]
   .sort_values("rank_test_score").head(10))

### 7.3 Choose the final model

The model with the best **validation** F1-macro is refitted on train + validation (more data, hyperparameters frozen) and evaluated once on test in section 8.

In [ ]:
results = results.sort_values("f1_macro", ascending=False)
BEST_NAME = results.index[0]
final_model = clone(fitted[BEST_NAME]).fit(pd.concat([X_train, X_val]), np.concatenate([y_train, y_val]))
baseline_model = fitted["Baseline (majority class)"]
print(f"Final model: {BEST_NAME} (validation F1-macro {results.loc[BEST_NAME, 'f1_macro']:.3f})")
results

**Observation.** ✏️ Chosen model, why, and the gain over the baseline.

<a id="s8"></a>
## 8. Evaluation on the test set

The test set has not been used for any decision so far, so these numbers estimate the performance on new data.

### 8.1 Test metrics

In [ ]:
test_scores = pd.DataFrame({"Baseline (majority class)": evaluate(baseline_model, X_test, y_test),
                            f"Final: {BEST_NAME}": evaluate(final_model, X_test, y_test)}).T
y_pred = final_model.predict(X_test)
display(test_scores)
pd.DataFrame(classification_report(y_test, y_pred, target_names=CLASS_NAMES, output_dict=True)).T

### 8.2 Confusion matrix

Rows = true class, columns = predicted class. The text shows the count and the share of the true class (row %), which reveals the recall of each class.

In [ ]:
cm = confusion_matrix(y_test, y_pred)
cm_row = cm / cm.sum(axis=1, keepdims=True) * 100
fig = go.Figure(go.Heatmap(z=cm_row, x=CLASS_NAMES, y=CLASS_NAMES, colorscale=SEQ_BLUE, zmin=0, zmax=100,
                           text=[[f"{n:,}<br>({p:.1f}%)" for n, p in zip(rn, rp)] for rn, rp in zip(cm, cm_row)],
                           texttemplate="%{text}", colorbar=dict(title="row %"),
                           hovertemplate="true %{y}, predicted %{x}: %{z:.1f}%<extra></extra>"))
show_fig(fig, f"Confusion matrix on test: recall of '{MINORITY}' = {cm_row[POS, POS]:.1f}%",
         xaxis_title="Predicted class", yaxis_title="True class", yaxis_autorange="reversed", height=460)

**Observation.** ✏️ Which classes are confused with which, and how often (numbers from the matrix).

### 8.3 ROC and precision–recall curves (binary tasks)

The ROC curve shows the trade-off between true and false positive rates over all thresholds; with imbalanced classes the precision–recall curve is more informative.

In [ ]:
if IS_BINARY:
    proba_pos = final_model.predict_proba(X_test)[:, POS]
    fpr, tpr, _ = roc_curve(y_test == POS, proba_pos)
    precision, recall, _ = precision_recall_curve(y_test == POS, proba_pos)
    fig = make_subplots(rows=1, cols=2, subplot_titles=(
        f"ROC (AUC = {roc_auc_score(y_test == POS, proba_pos):.3f})",
        f"Precision–recall (AP = {average_precision_score(y_test == POS, proba_pos):.3f})"))
    fig.add_trace(go.Scatter(x=fpr, y=tpr, mode="lines", line=dict(color=BLUE, width=2), name="ROC"), row=1, col=1)
    fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines", line=dict(color=MUTED, dash="dot"), name="random"), row=1, col=1)
    fig.add_trace(go.Scatter(x=recall, y=precision, mode="lines", line=dict(color=ORANGE, width=2), name="PR"), row=1, col=2)
    fig.add_hline(y=np.mean(y_test == POS), line_dash="dot", line_color=MUTED, row=1, col=2)
    fig.update_xaxes(title_text="False positive rate", row=1, col=1)
    fig.update_yaxes(title_text="True positive rate", row=1, col=1)
    fig.update_xaxes(title_text="Recall", row=1, col=2)
    fig.update_yaxes(title_text="Precision", row=1, col=2)
    show_fig(fig, f"Threshold-free view of the final model on test (positive class = '{MINORITY}')", height=440)
else:
    print("Multi-class target: see the per-class report in 8.1.")

### 8.4 Which features matter?

**Permutation importance**: shuffle one original column at a time on the test set and measure how much F1-macro drops. It works for any model and is computed on the original columns (before one-hot).

In [ ]:
perm = permutation_importance(final_model, X_test, y_test, scoring="f1_macro", n_repeats=5,
                              random_state=SEED, n_jobs=-1)
importance = (pd.DataFrame({"mean drop in F1-macro": perm.importances_mean, "std": perm.importances_std},
                           index=X_test.columns).sort_values("mean drop in F1-macro").tail(15))
fig = go.Figure(go.Bar(x=importance["mean drop in F1-macro"], y=importance.index, orientation="h",
                       error_x=dict(type="data", array=importance["std"], color=MUTED), marker_color=BLUE,
                       hovertemplate="%{y}: %{x:.4f}<extra></extra>"))
show_fig(fig, f"Permutation importance on test: '{importance.index[-1]}' matters most",
         xaxis_title="Drop in F1-macro when the column is shuffled", height=max(380, 30 * len(importance)))

**Observation.** ✏️ Top features; consistent with the EDA in section 5? (This is where EDA and model must agree.)

### 8.5 Error analysis

Wrongly predicted test rows, most confident mistakes first. ✏️ Look for a pattern (a sub-group, missing values, outliers) and write it down.

In [ ]:
proba = final_model.predict_proba(X_test)
errors = X_test.assign(true=label_encoder.inverse_transform(y_test),
                       predicted=label_encoder.inverse_transform(y_pred),
                       confidence=proba.max(axis=1))
errors = errors[errors["true"] != errors["predicted"]].sort_values("confidence", ascending=False)
print(f"{len(errors):,} wrong predictions out of {len(X_test):,} test rows ({len(errors) / len(X_test):.1%})")
errors.head(10)

**Observation.** ✏️

<a id="s9"></a>
## 9. Inference on new data

End-to-end use of the final pipeline: raw new rows → same text fixes as in 6.1 (`normalize_values`) → preprocessing → prediction. Columns that are missing in a new row become `NaN` and are imputed exactly like in training.

In [ ]:
# TODO: replace with hand-written rows, e.g. pd.DataFrame([{"age": 35, "city": " hcm", ...}])
new_rows = X_test.head(3).copy()

new_rows = normalize_values(new_rows).reindex(columns=FEATURES)
pred_proba = final_model.predict_proba(new_rows)
prediction = new_rows.assign(
    predicted=label_encoder.inverse_transform(pred_proba.argmax(axis=1)),
    **{f"P({c})": pred_proba[:, i] for i, c in enumerate(CLASS_NAMES)})
prediction

<a id="s10"></a>
## 10. Conclusion

### 10.1 Summary of results

✏️ Fill this table with the numbers printed in sections 7–8; the same numbers must appear in the PDF report and on the project page (`assets/js/data.js → results`).

| Model | Accuracy | Balanced accuracy | F1-macro | ROC-AUC |
|---|---|---|---|---|
| Baseline (majority class) | ✏️ | ✏️ | ✏️ | ✏️ |
| ✏️ final model | ✏️ | ✏️ | ✏️ | ✏️ |

### 10.2 What the analysis answered

✏️ 3–5 numbered findings, each with a number from this notebook and a link to its figure (e.g. "Figure 7 shows …").

### 10.3 Limitations

✏️ Data (size, period, sampling bias, label noise), method (assumptions, features not used), evaluation (single split, test period/distribution).

### 10.4 Extensions

✏️ Only list what you would actually try next. Ideas that were run and compared earn creativity points; ideas that were not run do not.

### 10.5 Submission checklist (delete before submitting)

- [ ] **Runtime → Restart and run all** on Google Colab finishes without errors, and the saved notebook keeps its outputs.
- [ ] No `TODO`, `✏️` or placeholder text is left (search the notebook for `TODO`).
- [ ] Every figure is Plotly, has a numbered conclusion-style title and labelled axes, and is followed by an observation that quotes numbers.
- [ ] Every number written in markdown matches the printed output of the current run.
- [ ] Imputers, scalers, encoders and vectorizers are fitted on **train only**; the test set is used once, in section 8.
- [ ] The model comparison includes a baseline, and the chosen model is selected on **validation**, not test.
- [ ] Cleaning rules are applied before the split; every `DROP_COLS` entry is justified.
- [ ] Section 2.3 shows the dataset meets assignment §4.2 (≥ 2,000 rows, ≥ 10 columns, missing values, categorical + numeric).
- [ ] `assets/js/data.js` is updated: notebook path, dataset, problem, results.